# EGFR-PACC OpenMM GPU-Relaxed Docking

Runtime -> Change runtime type -> T4 GPU, then run cells from top to bottom.

Upload `colab_egfr_pacc_md_input.zip` when prompted. Local package path:

`F:/egfr_pacc_digital_cell/reports/colab_md_relaxed_package/colab_egfr_pacc_md_input.zip`

This notebook uses `--platform auto`: OpenMM will use CUDA if registered; otherwise it falls back to OpenCL/CPU instead of crashing.

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install openmm pandas matplotlib tabulate
!apt-get -qq update
!apt-get -qq install -y openbabel wget unzip

In [ ]:
# Install AutoDock Vina Linux binary if `vina` is not already available.
import shutil
if shutil.which('vina') is None:
    !wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64 -O /usr/local/bin/vina
    !chmod +x /usr/local/bin/vina
!vina --help | head -20

In [ ]:
from openmm import Platform
print("OpenMM available platforms:")
for i in range(Platform.getNumPlatforms()):
    print(i, Platform.getPlatform(i).getName())
print("If CUDA is absent, the script will automatically use OpenCL/CPU instead of stopping.")

In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())

In [ ]:
!rm -rf colab_input colab_output colab_md_relaxed_egfr_pacc.py COLAB_GPU_MD_RELAXED_RUNBOOK_CN.md
!unzip -q colab_egfr_pacc_md_input.zip -d .
!find colab_input -maxdepth 3 -type f | sort
!ls -lh colab_md_relaxed_egfr_pacc.py

In [ ]:
# Quick smoke test: no MD steps and lower Vina exhaustiveness.
# This confirms that OpenMM, OpenBabel, Vina and the input package all work.
!python colab_md_relaxed_egfr_pacc.py \
  --input-dir colab_input \
  --output-dir colab_output/openmm_gpu_relaxed_quick \
  --platform auto \
  --max-iterations 200 \
  --md-steps 0 \
  --exhaustiveness 4

In [ ]:
# Formal run for revision-ready supplementary material.
!python colab_md_relaxed_egfr_pacc.py \
  --input-dir colab_input \
  --output-dir colab_output/openmm_gpu_relaxed \
  --platform auto \
  --max-iterations 500 \
  --md-steps 1000 \
  --exhaustiveness 8

In [ ]:
import pandas as pd
display(pd.read_csv('colab_output/openmm_gpu_relaxed/openmm_gpu_relaxed_receptor_metrics.csv'))
display(pd.read_csv('colab_output/openmm_gpu_relaxed/openmm_gpu_relaxed_docking_scores.csv'))
display(pd.read_csv('colab_output/openmm_gpu_relaxed/openmm_gpu_relaxed_vs_unrelaxed_comparison.csv'))

In [ ]:
from IPython.display import Image, display
display(Image('colab_output/openmm_gpu_relaxed/openmm_gpu_relaxed_docking_sensitivity.png'))

In [ ]:
!zip -qr openmm_gpu_relaxed_outputs.zip colab_output/openmm_gpu_relaxed
from google.colab import files
files.download('openmm_gpu_relaxed_outputs.zip')